### Join service areas generated by ArcGIS REST API

In [ ]:
tbl = 'X://escim/isocronas/merge_sadata.xlsx'

folder = 'X://escim/isocronas/'

outgeodb = 'X://escim/isocronas/arc_rest_iso.gdb'

In [ ]:
import os

from glass.pys.oss import lst_fld, fprop
from glass.pys.tm import now_as_str
from glass.rd import tbl_to_obj
from glass.esri.wenv      import create_geodb
from glass.esri.gp.gen import dissolve
from glass.esri.gp.ovl import erase
from glass.esri.dp import merge
from glass.esri.tbl.col import cols_calc

In [ ]:
outgeodb = create_geodb(
    os.path.dirname(outgeodb),
    os.path.basename(outgeodb)
)

tmpgdb = create_geodb(os.path.dirname(outgeodb), now_as_str())

In [ ]:
folders = lst_fld(folder, name=True, ignore_gdb=True)

In [ ]:
folders

In [ ]:
# For each folder
# Get files to be used to generate final service areas

data = {}
for f in folders:
    df = tbl_to_obj(tbl, sheet=f)
    
    df.sort_values(by=['order'], inplace=True)
    
    sas = df.filename.tolist()
    tmbreaks = df.time_interval.tolist()
    
    # Dissolve all shapes
    fsa = []
    for i in range(len(sas)):
        oname = fprop(sas[i], 'fn')
        dfc, dlyr = dissolve(
            os.path.join(folder, f, sas[i]),
            os.path.join(tmpgdb, f'diss_{oname}'),
            "",
            geomMultiPart=None
        )
        
        # If first shp, stop
        if not i:
            # Update Field
            cols_calc(dfc, 'tmbreaks', f"'{tmbreaks[i]}'")
            
            fsa.append(dfc)
            
            continue
        
        # Erase
        erasa, eralyr = erase(
            dfc,
            os.path.join(folder, f, sas[i-1]),
            os.path.join(tmpgdb, f'era_{oname}')
        )
        
        # Update Field
        cols_calc(erasa, 'tmbreaks', f"'{tmbreaks[i]}'")
        
        fsa.append(erasa)
    
    # Merge
    data[f], sa_lyr = merge(fsa, os.path.join(outgeodb, f))

In [ ]:
data